# Person A — Notebook 2: Ablation and Final Results
**CS 590NN | Amogh | Week 3 (Mar 24-30)**

Runs the 1/5/10-step ablation and produces harness-ready output for Person B.

**Before running:**
- Upload `week2_circuit_log.json` from Notebook 1 using the upload cell below

> Runtime: L4 GPU (Colab Pro)
> Start a fresh session — do not reuse Notebook 1's session.

### 2.0 Install
Run once. Runtime restarts automatically.

In [ ]:
import subprocess, sys, os

def pip(args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + args)

pip(["numpy==1.26.4"])
pip(["transformer-lens","transformers","datasets","accelerate","einops","jaxtyping"])

print("Done. Restarting runtime...")
os.kill(os.getpid(), 9)

### 2.1 Imports
Start here after restart.

In [1]:
import torch, json, re, requests
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from transformer_lens import HookedTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"Memory : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

import numpy as np
assert tuple(int(x) for x in np.__version__.split(".")[:2]) < (2,0),     "NumPy >= 2.0 — re-run cell 2.0 and restart."

torch.manual_seed(42)
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
print("Imports OK")

Device : cuda
GPU    : NVIDIA L4
Memory : 23.5 GB free / 23.7 GB total
Imports OK


### 2.2 Load Model

In [2]:
MODEL_NAME_PRIMARY  = "Qwen/Qwen3-0.6B"
MODEL_NAME_FALLBACK = "Qwen/Qwen2.5-0.5B"

def load_model(name):
    return HookedTransformer.from_pretrained(
        name,
        center_unembed=True,
        center_writing_weights=False,
        fold_ln=True,
        refactor_factored_attn_matrices=False,
        device=DEVICE,
    )

try:
    model = load_model(MODEL_NAME_PRIMARY)
    MODEL_NAME = MODEL_NAME_PRIMARY
except Exception as e:
    print(f"Primary failed ({e}). Using fallback...")
    model = load_model(MODEL_NAME_FALLBACK)
    MODEL_NAME = MODEL_NAME_FALLBACK

model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token
free, _ = torch.cuda.mem_get_info()
print(f"Loaded : {MODEL_NAME}")
print(f"Memory : {free/1e9:.1f} GB free after load")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Loaded pretrained model Qwen/Qwen3-0.6B into HookedTransformer
Loaded : Qwen/Qwen3-0.6B
Memory : 20.3 GB free after load


### 2.3 Upload Circuit Log from Notebook 1

In [3]:
from google.colab import files

print("Upload week2_circuit_log.json from Notebook 1...")
uploaded = files.upload()

# Move to results dir
import shutil
for fname in uploaded:
    shutil.move(fname, RESULTS_DIR / fname)
    print(f"Saved -> {RESULTS_DIR / fname}")

with open(RESULTS_DIR / "week2_circuit_log.json") as f:
    circuit_log = json.load(f)

N_SAMPLES = len(circuit_log)
print(f"Circuit log loaded: {N_SAMPLES} entries")

Upload week2_circuit_log.json from Notebook 1...


Saving week2_circuit_log.json to week2_circuit_log.json
Saved -> results/week2_circuit_log.json
Circuit log loaded: 50 entries


### 2.4 Load CounterFact

In [4]:
@dataclass
class EditSample:
    prompt:          str
    target_new:      str
    target_true:     str
    related_prompts: List[str] = field(default_factory=list)

print("Downloading CounterFact...")
raw_data = requests.get("https://rome.baulab.info/data/dsets/counterfact.json", timeout=60).json()

def parse_counterfact(raw):
    neighborhood = raw.get("neighborhood_prompts", [])
    related = [p for p in neighborhood[:3] if isinstance(p, str)]
    return EditSample(
        prompt=raw["requested_rewrite"]["prompt"].format(raw["requested_rewrite"]["subject"]),
        target_new=" "  + raw["requested_rewrite"]["target_new"]["str"],
        target_true=" " + raw["requested_rewrite"]["target_true"]["str"],
        related_prompts=related,
    )

cf_samples = [parse_counterfact(raw_data[i]) for i in range(N_SAMPLES)]
print(f"Loaded {N_SAMPLES} samples")

Loaded 50 samples


### 2.5 Edit Functions

In [5]:
def restore_weights(model, state):
    """Restore weights in-place — no temporary GPU allocation."""
    with torch.no_grad():
        for name, param in model.named_parameters():
            param.copy_(state[name])
    torch.cuda.empty_cache()


def get_circuit_params(model, circuit_attn, circuit_mlp):
    params = []
    for (layer, head) in circuit_attn:
        try: params.append(model.blocks[layer].attn.W_O)
        except: pass
    for layer in circuit_mlp:
        try:
            params.append(model.blocks[layer].mlp.W_in)
            params.append(model.blocks[layer].mlp.W_out)
        except: pass
    return params


def contrastive_loss(model, sample):
    tokens  = model.to_tokens(sample.prompt)
    new_id  = model.to_tokens(sample.target_new,  prepend_bos=False)[0, 0]
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0, 0]
    logits  = model(tokens)
    lp      = torch.nn.functional.log_softmax(logits[0, -1, :], dim=-1)
    loss    = -lp[new_id] + lp[true_id]
    del tokens, logits
    return loss, lp[new_id].exp().item(), lp[true_id].exp().item()


def run_edit(model, sample, circuit_attn, circuit_mlp, n_steps, lr=5e-3):
    params = get_circuit_params(model, circuit_attn, circuit_mlp)
    if not params:
        params = [p for layer in model.blocks
                  for p in [layer.mlp.W_in, layer.mlp.W_out]]

    for p in model.parameters(): p.requires_grad_(False)
    for p in params:              p.requires_grad_(True)

    optimizer = torch.optim.Adam(params, lr=lr)

    model.eval()
    with torch.no_grad():
        _, baseline, _ = contrastive_loss(model, sample)

    for _ in range(n_steps):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss, _, _ = contrastive_loss(model, sample)
        loss.backward()
        optimizer.step()
        del loss

    torch.cuda.empty_cache()
    model.eval()
    with torch.no_grad():
        _, final, final_true = contrastive_loss(model, sample)

    for p in model.parameters(): p.requires_grad_(True)
    return {"edit_success": final, "baseline_prob": baseline,
            "final_p_true": final_true, "kl_final": 0.0}


# Save original weights to CPU once
original_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
torch.cuda.empty_cache()
free, _ = torch.cuda.mem_get_info()
print(f"Original weights saved to CPU")
print(f"GPU free: {free/1e9:.1f} GB — ready for ablation")

Original weights saved to CPU
GPU free: 20.3 GB — ready for ablation


### 2.6 Step-Count Ablation: 1 / 5 / 10 Steps
Primary result table for the paper.

In [6]:
STEP_COUNTS      = [1, 5, 10]
LR               = 5e-3
ablation_results = {s: [] for s in STEP_COUNTS}

for n_steps in STEP_COUNTS:
    print(f"\n{n_steps}-step ablation on {N_SAMPLES} samples")
    for i in range(N_SAMPLES):
        restore_weights(model, original_state)
        res = run_edit(
            model, cf_samples[i],
            circuit_log[i]["attn_heads"],
            circuit_log[i]["mlp_layers"],
            n_steps=n_steps, lr=LR,
        )
        res["idx"]     = i
        res["n_steps"] = n_steps
        ablation_results[n_steps].append(res)

        if i % 10 == 0:
            free = torch.cuda.mem_get_info()[0] / 1e9
            print(f"  [{i+1:>2}/{N_SAMPLES}]  "
                  f"delta={res['edit_success']-res['baseline_prob']:+.4f}  "
                  f"gpu_free={free:.1f}GB")

    avg_e = sum(r["edit_success"]  for r in ablation_results[n_steps]) / N_SAMPLES
    avg_b = sum(r["baseline_prob"] for r in ablation_results[n_steps]) / N_SAMPLES
    print(f"  SUMMARY  baseline={avg_b:.4f}  edit={avg_e:.4f}  delta={avg_e-avg_b:+.4f}")

restore_weights(model, original_state)

with open(RESULTS_DIR / "week3_ablation.json", "w") as f:
    json.dump({str(k): v for k, v in ablation_results.items()}, f, indent=2)
print(f"\nSaved -> {RESULTS_DIR}/week3_ablation.json")


1-step ablation on 50 samples
  [ 1/50]  delta=+0.6430  gpu_free=19.2GB
  [11/50]  delta=+1.0000  gpu_free=18.4GB
  [21/50]  delta=-0.0006  gpu_free=18.0GB
  [31/50]  delta=+0.9980  gpu_free=17.9GB
  [41/50]  delta=+0.3963  gpu_free=18.4GB
  SUMMARY  baseline=0.0032  edit=0.4619  delta=+0.4587

5-step ablation on 50 samples
  [ 1/50]  delta=+0.9982  gpu_free=18.8GB
  [11/50]  delta=+1.0000  gpu_free=18.6GB
  [21/50]  delta=+0.9994  gpu_free=18.2GB
  [31/50]  delta=+0.9984  gpu_free=18.1GB
  [41/50]  delta=+1.0000  gpu_free=18.6GB
  SUMMARY  baseline=0.0032  edit=0.9788  delta=+0.9756

10-step ablation on 50 samples
  [ 1/50]  delta=+0.9982  gpu_free=18.9GB
  [11/50]  delta=+1.0000  gpu_free=18.6GB
  [21/50]  delta=+0.9994  gpu_free=18.2GB
  [31/50]  delta=+0.9984  gpu_free=18.1GB
  [41/50]  delta=+1.0000  gpu_free=18.6GB
  SUMMARY  baseline=0.0032  edit=0.9979  delta=+0.9947

Saved -> results/week3_ablation.json


### 2.7 Harness-Ready Output for Person B

In [7]:
rows, summary = [], {}
for n_steps, results in ablation_results.items():
    key = f"OurMethod_{n_steps}step"
    for r in results:
        rows.append({
            "method":         key,
            "model":          MODEL_NAME,
            "idx":            r["idx"],
            "n_steps":        n_steps,
            "edit_success":   round(r["edit_success"],  4),
            "baseline_prob":  round(r["baseline_prob"], 4),
            "over_extinction": None,
            "kl_final":       round(r["kl_final"], 4),
        })
    summary[key] = {
        "avg_edit_success": round(sum(r["edit_success"]  for r in results) / len(results), 4),
        "avg_baseline":     round(sum(r["baseline_prob"] for r in results) / len(results), 4),
        "avg_delta":        round(sum(r["edit_success"]-r["baseline_prob"] for r in results) / len(results), 4),
        "n_samples":        len(results),
    }

harness_output = {"model": MODEL_NAME, "rows": rows, "summary": summary}
with open(RESULTS_DIR / "week3_harness_output.json", "w") as f:
    json.dump(harness_output, f, indent=2)

print(f"{'Method':<28}  {'edit':>6}  {'baseline':>8}  {'delta':>7}")
print("-" * 54)
for method, s in summary.items():
    print(f"{method:<28}  {s['avg_edit_success']:>6.4f}  "
          f"{s['avg_baseline']:>8.4f}  {s['avg_delta']:>+7.4f}")
print(f"\nShare week3_harness_output.json with Person B at Sync Point 2 (Mar 24 EOD).")

Method                          edit  baseline    delta
------------------------------------------------------
OurMethod_1step               0.4619    0.0032  +0.4587
OurMethod_5step               0.9788    0.0032  +0.9756
OurMethod_10step              0.9979    0.0032  +0.9947

Share week3_harness_output.json with Person B at Sync Point 2 (Mar 24 EOD).


### 2.8 Save All Results and Download

In [8]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Master summary
master_summary = {
    "author":    "Amogh",
    "notebook":  "Notebook 2 — Ablation and Final Results",
    "model":     MODEL_NAME,
    "timestamp": timestamp,
    "n_samples": N_SAMPLES,
    "hyperparams": {"lr": LR, "step_counts": STEP_COUNTS},
    "ablation_summary": {},
}

for n_steps, results in ablation_results.items():
    avg_e = sum(r["edit_success"]  for r in results) / len(results)
    avg_b = sum(r["baseline_prob"] for r in results) / len(results)
    master_summary["ablation_summary"][f"{n_steps}_step"] = {
        "avg_baseline":     round(avg_b, 4),
        "avg_edit_success": round(avg_e, 4),
        "avg_delta":        round(avg_e - avg_b, 4),
    }

with open(RESULTS_DIR / f"summary_nb2_{timestamp}.json", "w") as f:
    json.dump(master_summary, f, indent=2)

zip_path = f"/content/PersonA_Notebook2_results_{timestamp}"
shutil.make_archive(zip_path, "zip", RESULTS_DIR)

# Print final table
print("=" * 60)
print(f"  FINAL RESULTS — Amogh | {MODEL_NAME}")
print(f"  {timestamp}")
print("=" * 60)
print(f"\n  {'Steps':<8}  {'Baseline':>8}  {'Edit':>8}  {'Delta':>8}")
print(f"  {'-'*40}")
for k, v in master_summary["ablation_summary"].items():
    print(f"  {k:<8}  {v['avg_baseline']:>8.4f}  "
          f"{v['avg_edit_success']:>8.4f}  {v['avg_delta']:>+8.4f}")
print()
print("  Files saved:")
for f in sorted(RESULTS_DIR.glob("*.json")):
    print(f"    {f.name:<45}  {f.stat().st_size//1024:>4} KB")
print(f"\n  Download: {zip_path}.zip")
print("=" * 60)

from google.colab import files
files.download(f"{zip_path}.zip")

  FINAL RESULTS — Amogh | Qwen/Qwen3-0.6B
  20260310_172801

  Steps     Baseline      Edit     Delta
  ----------------------------------------
  1_step      0.0032    0.4619   +0.4587
  5_step      0.0032    0.9788   +0.9756
  10_step     0.0032    0.9979   +0.9947

  Files saved:
    summary_nb2_20260310_172801.json                  0 KB
    week2_circuit_log.json                           20 KB
    week3_ablation.json                              28 KB
    week3_harness_output.json                        34 KB

  Download: /content/PersonA_Notebook2_results_20260310_172801.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>